<a href="https://colab.research.google.com/github/faisalepty/Sign-Language-CNN/blob/main/finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torchvision import models, transforms, datasets
from tqdm import tqdm


In [2]:
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, shear=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
import kagglehub

path = kagglehub.dataset_download("kapillondhe/american-sign-language", output_dir="./data")

print("Path to dataset files:", path)

100%|██████████| 4.64G/4.64G [01:03<00:00, 78.6MB/s]

Extracting files...


Path to dataset files: ./data


In [ ]:
path1 = kagglehub.dataset_download("grassknoted/asl-alphabet")

print("Path to dataset files:", path1)

Using Colab cache for faster access to the 'asl-alphabet' dataset.
Path to dataset files: /kaggle/input/asl-alphabet


In [ ]:
path2 = kagglehub.dataset_download("debashishsau/aslamerican-sign-language-aplhabet-dataset")

print("Path to dataset files:", path2)

100%|██████████| 4.20G/4.20G [00:46<00:00, 96.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/debashishsau/aslamerican-sign-language-aplhabet-dataset/versions/1


In [ ]:
path3 = kagglehub.dataset_download("mrgeislinger/asl-rgb-depth-fingerspelling-spelling-it-out", output_dir="./'''/data/")

print("Path to dataset files:", path3)

100%|██████████| 2.11G/2.11G [00:27<00:00, 82.6MB/s]

Extracting files...


Path to dataset files: ./'''/data/


In [ ]:
path4 = kagglehub.dataset_download("ahmedkhanak1995/sign-language-gesture-images-dataset", output_dir="'/data4")

print("Path to dataset files:", path4)

100%|██████████| 191M/191M [00:01<00:00, 141MB/s]

Extracting files...


Path to dataset files: '/data4


In [ ]:
path5 = kagglehub.dataset_download("alhasangamalmahmoud/american-sign-language-asl", output_dir="''/data5")

print("Path to dataset files:", path5)

100%|██████████| 2.09G/2.09G [00:33<00:00, 67.9MB/s]

Extracting files...


Path to dataset files: ''/data5


In [ ]:
path6 = kagglehub.dataset_download("nitinkumar00/american-sign-language-asl", output_dir="./''''/test1")

print("Path to dataset files:", path6)

100%|██████████| 5.90G/5.90G [01:20<00:00, 78.8MB/s]

Extracting files...


Path to dataset files: ./''''/test1


In [ ]:
path7 = kagglehub.dataset_download("lexset/synthetic-asl-alphabet", output_dir="./'''''/test_data")

print("Path to dataset files:", path7)

100%|██████████| 6.58G/6.58G [01:22<00:00, 85.3MB/s]

Extracting files...


Path to dataset files: ./'''''/test_data


In [ ]:
path8 = kagglehub.dataset_download("rahulmakwana/american-sign-language-recognition", output_dir="./''''''/data")

print("Path to dataset files:", path8)

Using Colab cache for faster access to the 'american-sign-language-recognition' dataset.
Path to dataset files: /kaggle/input/american-sign-language-recognition


In [ ]:
path9 = kagglehub.dataset_download("dorukdemirci/asl-alphabet-dataset", output_dir="./'''''''/test_data")

print("Path to dataset files:", path9)

Using Colab cache for faster access to the 'asl-alphabet-dataset' dataset.
Path to dataset files: /kaggle/input/asl-alphabet-dataset


In [ ]:
import shutil
import os

# Define the base source and destination directories
base_source_dir = "/kaggle/input/asl-alphabet-dataset/dataset"
base_destination_dir = '/content/data/ASL_Dataset/Train'

# Define the letters to iterate through
letters = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

for letter in letters:
    # Construct source path with lowercase subfolder name
    source_dir = os.path.join(base_source_dir, letter.upper() + "-samples")
    # Destination path remains with uppercase letter
    destination_dir = os.path.join(base_destination_dir, letter.upper())

    # Create the parent directory for the destination if it doesn't exist
    os.makedirs(os.path.dirname(destination_dir), exist_ok=True)

    try:
        # Use copytree to copy the di
        shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
        print(f"Successfully copied '{source_dir}' to '{destination_dir}'")
    except FileNotFoundError:
        print(f"Error: Source directory '{source_dir}' not found. Skipping {letter}")
    except Exception as e:
        print(f"An error occurred during copying '{source_dir}': {e}")


Successfully copied '/kaggle/input/asl-alphabet-dataset/dataset/A-samples' to '/content/data/ASL_Dataset/Train/A'
Successfully copied '/kaggle/input/asl-alphabet-dataset/dataset/B-samples' to '/content/data/ASL_Dataset/Train/B'
Successfully copied '/kaggle/input/asl-alphabet-dataset/dataset/C-samples' to '/content/data/ASL_Dataset/Train/C'
Successfully copied '/kaggle/input/asl-alphabet-dataset/dataset/D-samples' to '/content/data/ASL_Dataset/Train/D'
Successfully copied '/kaggle/input/asl-alphabet-dataset/dataset/E-samples' to '/content/data/ASL_Dataset/Train/E'
Successfully copied '/kaggle/input/asl-alphabet-dataset/dataset/F-samples' to '/content/data/ASL_Dataset/Train/F'
Successfully copied '/kaggle/input/asl-alphabet-dataset/dataset/G-samples' to '/content/data/ASL_Dataset/Train/G'
Successfully copied '/kaggle/input/asl-alphabet-dataset/dataset/H-samples' to '/content/data/ASL_Dataset/Train/H'
Successfully copied '/kaggle/input/asl-alphabet-dataset/dataset/I-samples' to '/content/

In [ ]:
import shutil
shutil.rmtree("/content/data/ASL_Dataset/Train/.ipynb_checkpoints", ignore_errors=True)

In [ ]:
path = "data"
dataset2 = datasets.ImageFolder("/content/data/ASL_Dataset/Train", transform=val_transforms)
dataset1 = datasets.ImageFolder("/root/.cache/kagglehub/datasets/debashishsau/aslamerican-sign-language-aplhabet-dataset/versions/1/ASL_Alphabet_Dataset/asl_alphabet_train", transform=val_transforms)
dataset3 = datasets.ImageFolder("/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train", transform=val_transforms)


dataset = ConcatDataset([dataset1, dataset2, dataset3])

n_total = len(dataset)
n_train = int(0.8*n_total)
n_val = int(n_total - n_train)

train_dataset, val_dataset = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("krishnapaanchajanya/american-sign-language-asl", output_dir="./data")

print("Path to dataset files:", path)

100%|██████████| 332M/332M [00:23<00:00, 14.5MB/s]

Extracting files...


Path to dataset files: ./data


In [20]:
path = kagglehub.dataset_download("kuzivakwashe/significant-asl-sign-language-alphabet-dataset", output_dir="./data2")

print("Path to dataset files:", path)

100%|██████████| 3.88G/3.88G [03:55<00:00, 17.7MB/s]

Extracting files...


Path to dataset files: ./data2


In [26]:
data = datasets.ImageFolder('/content/data3/American Sign Language (ASL)', transform=val_transforms)
test_loader = DataLoader(data, batch_size=128, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
print('Classes in dataset1:', data.classes)

Classes in dataset1: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [25]:
# Download latest version
path = kagglehub.dataset_download("abhishekradadiya/american-sign-language-asl", output_dir="./data3")

print("Path to dataset files:", path)

100%|██████████| 15.8M/15.8M [00:02<00:00, 6.64MB/s]

Extracting files...


Path to dataset files: ./data3


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_dataset,   batch_size=128, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
print('Classes in dataset1:', dataset1.classes)
print('Classes in dataset2:', dataset2.classes)
print('Classes in dataset3:', dataset3.classes)

Classes in dataset1: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Classes in dataset2: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Classes in dataset3: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [5]:
from torch.nn.modules.linear import Linear
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 29)
)

model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 69.5MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
device = ("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
model.load_state_dict(torch.load("/content/drive/MyDrive/Copy of finetune_modelv2.pth", map_location=torch.device(device)))

<All keys matched successfully>

In [ ]:
for param in model.parameters():
  param.requires_grad = True

In [10]:
optimizer = torch.optim.AdamW(
    [{'params': model.features.parameters(),    'lr': 1e-5},  # backbone
    {'params': model.classifier.parameters(),  'lr': 1e-4}   # classifier
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler    = GradScaler()

/tmp/ipykernel_580/4038444618.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()


In [ ]:
def train(model, loader, optimiser, scaler, device):
  model.train()
  running_loss = 0.0

  for images, labels in tqdm(loader, desc="training"):
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()

    with autocast():
      outputs = model(images)
      loss = criterion(outputs, labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    running_loss += loss.item()

  return running_loss / len(loader)

In [11]:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)

            with autocast():
                outputs = model(images)
                loss    = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted  = outputs.max(1)
            correct       += predicted.eq(labels).sum().item()
            total         += labels.size(0)

    return running_loss / len(loader), correct / total

In [28]:
from PIL import Image
import torch.nn.functional as F

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

def predict(image_tensor, model, device, true_label=None):
    # Set model to evaluation mode
    model.eval()

    # Add batch dimension if it's a single image
    if image_tensor.ndim == 3:
        image_tensor = image_tensor.unsqueeze(0)

    image_tensor = image_tensor.to(device)

    with torch.no_grad():
        output = model(image_tensor)
        # Apply softmax to get probabilities
        probabilities = F.softmax(output, dim=1)
        # Get the predicted class index and confidence
        conf, predicted_idx = torch.max(probabilities, 1)

    # Map index to label
    labels = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J',
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T',
    'U', 'V', 'W', 'X', 'Y', 'Z',
    'del', 'nothing', 'space'
    ]

    prediction_text = f"Prediction: {labels[predicted_idx.item()]} ({conf.item()*100:.2f}%%)"
    if true_label is not None:
        prediction_text += f" | Actual: {labels[true_label.item()]}"
    print(prediction_text)


In [ ]:
model.to(device)
epochs = 7
best_val_acc = 0.0
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimiser=optimizer, scaler=scaler, device=device)
    val_loss, val_acc = validate(model, val_loader, criterion=criterion, device=device)
    scheduler.step()

    if val_acc > best_val_acc:
      best_val_acc = val_acc
      torch.save(model.state_dict(), f"/content/drive/My Drive/finetune_modelv{epoch}.pth")



    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

training:   0%|          | 0/3760 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_10092/2826028204.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validating:   0%|          | 0/940 [00:00<?, ?it/s]/tmp/ipykernel_10092/1206446172.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validating: 100%|██████████| 940/940 [10:46<00:00,  1.46it/

Epoch [1/7], Train Loss: 0.0923, Val Loss: 0.0231, Val Acc: 0.9924


Validating: 100%|██████████| 940/940 [10:22<00:00,  1.51it/s]


Epoch [2/7], Train Loss: 0.0419, Val Loss: 0.0233, Val Acc: 0.9925


Validating: 100%|██████████| 940/940 [10:12<00:00,  1.53it/s]


Epoch [3/7], Train Loss: 0.0324, Val Loss: 0.0122, Val Acc: 0.9960


training:  89%|████████▉ | 3347/3760 [40:03<03:42,  1.86it/s]

In [34]:
 validate(model, test_loader, criterion=criterion, device=device)

Validating:   0%|          | 0/7 [00:00<?, ?it/s]/tmp/ipykernel_580/1206446172.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validating: 100%|██████████| 7/7 [00:03<00:00,  1.95it/s]


(3.368137836456299, 0.7628205128205128)

### Dataset Analysis for Discrepancy

Let's check the sizes and class distributions of the training, validation, and test datasets to identify potential issues.

In [38]:
# Test a few images from the test_loader
num_samples_to_test = 5
processed_samples = 0

for images_batch, labels_batch in test_loader:
    for i in range(images_batch.shape[0]): # Iterate through images in the batch
        # if processed_samples >= num_samples_to_test:
            # break
        print(f"--- Sample {processed_samples + 1} ---")
        predict(images_batch[i], model, device, labels_batch[i])
        processed_samples += 1
    if processed_samples >= num_samples_to_test:
        break


--- Sample 1 ---
Prediction: U (99.88%%) | Actual: F
--- Sample 2 ---
Prediction: G (92.18%%) | Actual: Z
--- Sample 3 ---
Prediction: J (99.85%%) | Actual: J
--- Sample 4 ---
Prediction: W (99.99%%) | Actual: W
--- Sample 5 ---
Prediction: R (100.00%%) | Actual: R
--- Sample 6 ---
Prediction: C (100.00%%) | Actual: C
--- Sample 7 ---
Prediction: L (100.00%%) | Actual: L
--- Sample 8 ---
Prediction: I (61.91%%) | Actual: N
--- Sample 9 ---
Prediction: A (100.00%%) | Actual: A
--- Sample 10 ---
Prediction: O (100.00%%) | Actual: O
--- Sample 11 ---
Prediction: K (92.17%%) | Actual: K
--- Sample 12 ---
Prediction: U (100.00%%) | Actual: U
--- Sample 13 ---
Prediction: J (100.00%%) | Actual: J
--- Sample 14 ---
Prediction: E (100.00%%) | Actual: E
--- Sample 15 ---
Prediction: N (67.76%%) | Actual: N
--- Sample 16 ---
Prediction: Q (100.00%%) | Actual: Q
--- Sample 17 ---
Prediction: G (99.94%%) | Actual: G
--- Sample 18 ---
Prediction: Y (99.63%%) | Actual: Y
--- Sample 19 ---
Prediction

In [ ]:
print(f"Number of samples in training dataset: {len(train_dataset)}")
print(f"Number of samples in validation dataset: {len(val_dataset)}")
print(f"Number of samples in test dataset: {len(data)}")